# Held-out IDSSE benchmark

This notebook validates the possession models on **unseen IDSSE matches**, not on the public BAR-ATH ground truth that was used for model selection.

It runs two panels:

- **Panel A, fair IDSSE-only comparison:** nearest-player baseline, physical HMM, Bundesliga-trained HMM, HMM ensemble.
- **Panel B, BAR-ATH transfer:** TCN and consensus. IDSSE has no Smart Tagging, so the set-piece features are neutralised.

> `J03WMX` is used to train `models/bundesliga_weights.json`. Never use it as a held-out match. The default matches are `J03WN1` and `J03WPY`, both unseen.

Runtime: use a CPU runtime. The two position files are about 390 MB each, and the full run usually takes a few minutes.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "evaluation" / "possession_metrics.py").exists():
    ROOT = ROOT.parent
if not (ROOT / "evaluation" / "possession_metrics.py").exists():
    matches = list(Path("/content").glob("*/evaluation/possession_metrics.py"))
    if matches:
        ROOT = matches[0].parents[1]
if not (ROOT / "evaluation" / "possession_metrics.py").exists():
    raise RuntimeError("Open this notebook from a clone of the repository, or clone it into /content first.")
sys.path.insert(0, str(ROOT / "models"))
print(ROOT)

## Configuration

Keep `download=True` on Colab. The module downloads the two XML files per match into `/content/idsse` and writes results into `/content/heldout_out`.

If you want a faster smoke run, set `MATCHES = ["J03WN1"]` and `BOOTSTRAP = 250` first, then rerun with both matches and 1000 draws.

In [ ]:
from heldout_idesse import run_benchmark

MATCHES = ["J03WN1", "J03WPY"]
DATA_DIR = Path("/content/idsse")
OUT_DIR = Path("/content/heldout_out")
BOOTSTRAP = 1000

summary = run_benchmark(
    matches=MATCHES,
    data_dir=DATA_DIR,
    out_dir=OUT_DIR,
    bootstrap=BOOTSTRAP,
    download=True,
)

## Per-match results

`spell_f1` is the ranking metric. `spell_lo` and `spell_hi` are the 5-95% bootstrap interval over ground-truth possessions. `macro_f1` is the frame-level tie-break.

In [ ]:
summary["per_match"]

## Pooled results

The pooled row concatenates the matches with an X separator, so no possession or transition is scored across matches.

In [ ]:
summary["pooled"]

## Paired differences

Positive `difference` means model A beats model B. `fraction_a_better` is the fraction of paired bootstrap draws where A wins. The key row is `consensus` minus `tcn_clean`.

In [ ]:
summary["paired"]

## Plot

In [ ]:
import matplotlib.pyplot as plt

order = ["hmm_physical", "hmm_bundesliga", "hmm_ensemble", "tcn_clean", "consensus"]
frame = summary["per_match"]
pivot = frame.pivot(index="model", columns="match", values="spell_f1").reindex(order)
ax = pivot.plot.bar(figsize=(10, 4), rot=30)
ax.set_ylabel("spell F1")
ax.set_title("Held-out IDSSE spell F1")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## How to read this

- Panel A is the clean comparison: all models see the same IDSSE-only information.
- Panel B is a transfer test. The TCN and consensus were trained/tuned on BAR-ATH and receive neutralised set-piece features here, so a drop is expected. A win is still meaningful; a loss is not proof that they are worse on the private BAR-ATH match.
- The team ran this benchmark after freezing the submission. `consensus` loses to `tcn_clean` in both unseen matches (pooled spellF1 0.3252 vs 0.3415, paired difference -0.0163, 95% CI [-0.0271, -0.0043]). That is the expected cost of tuning the consensus constants on the public BAR-ATH window. The submission stays `los_chuchos.csv` (consensus) for its public scoreboard position, and the held-out gap is declared as a limitation in `TECHNICAL_DOCUMENT.md`.
- Do not compare these absolute scores with the public BAR-ATH scoreboard: the matches, camera data and label coverage differ.

The full JSON, CSV and markdown report are in `OUT_DIR`.